# 1. Perkenalan Dataset

Pada proyek ini, digunakan Student Lifestyle and Stress Prediction Dataset yang diperoleh dari Kaggle. Dataset ini berisi informasi mengenai gaya hidup, dukungan sosial, dan karakteristik mahasiswa yang digunakan untuk memprediksi tingkat stres.

Dataset ini memiliki total sekitar 25.500 baris dengan fitur-fitur berikut:
- `Student_Type`: tipe mahasiswa (misalnya `school`, `college`, `working_student`)
- `Sleep_Hours`: jumlah jam tidur per hari
- `Study_Hours`: jumlah jam belajar per hari
- `Social_Media_Hours`: waktu penggunaan media sosial per hari
- `Attendance`: persentase kehadiran kuliah
- `Exam_Pressure`: tingkat tekanan ujian
- `Family_Support`: tingkat dukungan keluarga
- `Month`: bulan pengambilan data

Target pada dataset ini berupa klasifikasi biner (`Stress_Level`) yang menunjukkan apakah mahasiswa mengalami:
- `0`: stres rendah
- `1`: stres tinggi

Dataset dapat diakses melalui tautan berikut:
https://www.kaggle.com/datasets/sridevilavanyacse/student-lifestyle-and-stress-prediction-dataset/data

# **2. Import Library**

In [5]:
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix
)
import collections
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from imblearn.under_sampling import RandomUnderSampler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.model_selection import train_test_split

ImportError: sklearn._cyutility does not export expected C function slice_memviewslice

# **3. Memuat Dataset**

In [ ]:
df = pd.read_csv('../data_raw/student-lifestyle-and-stress-dataset.csv')
df.head()

# **4. Exploratory Data Analysis (EDA)**

Pada tahap ini dilakukan eksplorasi data untuk memahami karakteristik dataset, distribusi fitur, hubungan antar variabel, serta mendeteksi potensi permasalahan seperti missing value, outlier, dan ketidakseimbangan kelas.

Exploratory Data Analysis (EDA) membantu dalam menentukan strategi preprocessing dan pemilihan model machine learning yang tepat.

In [ ]:
# 1. Memeriksa dimensi dataset (jumlah baris dan kolom)
print("Dimensi Dataset:", df.shape)

# 2. Memeriksa tipe data dan informasi umum setiap kolom
print("\n--- Informasi Dataset ---")
df.info()

# 3. Melihat statistik deskriptif untuk fitur numerik
print("\n--- Statistik Deskriptif ---")
df.describe()

In [ ]:
# Mengecek jumlah missing values di setiap kolom
print("Jumlah Missing Values:")
print(df.isnull().sum())

# Mengecek jumlah baris yang duplikat
print("\nJumlah Data Duplikat:", df.duplicated().sum())

In [ ]:
sns.countplot(data=df, x='Student_Type', hue='Student_Type', palette='Set2', legend=False)
plt.title('Distribusi Kategori Mahasiswa')
plt.show()

In [ ]:
#Memilih kolom numerik
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns

#+# MULAI CODE ###

# Hitung matriks korelasi
correlation = df[numerical_cols].corr()

# Buat visualisasi heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(correlation,
               annot=True,
               cmap='coolwarm',
               fmt=".2f",
               vmin=-1,
               vmax=1)
plt.title('Correlation Matrix')
plt.show()

### SELESAI CODE ###

In [ ]:
df.hist(figsize=(15,12))

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

# --- PERBAIKAN DI SINI ---
# Pastikan semua kolom yang seharusnya angka dikonversi dulu di DataFrame asli.
# Ini opsional, tapi sangat disarankan agar sinkron dengan visualisasi.
for col in df.columns:
    if col != "Stress_Level":  # Hindari mengubah target jika berupa kategori nominal/integer khusus
        # Mencoba konversi, jika berhasil sebagian (karena mayoritas angka), kolom akan diupdate
        converted = pd.to_numeric(df[col], errors="coerce")
        # Jika setelah dikonversi persentase missing value tidak melonjak ekstrem, kita pakai hasil konversinya
        if converted.notnull().sum() > (0.5 * len(df)):
            df[col] = converted

# 1. Mengambil semua kolom bertipe angka setelah DataFrame dipastikan bersih
numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

if "Stress_Level" in numerical_cols:
    numerical_cols.remove("Stress_Level")

# 2. Atur ukuran grid secara dinamis berdasarkan jumlah kolom numerik
num_features = len(numerical_cols)
num_rows = (num_features + 2) // 3

fig, axes = plt.subplots(num_rows, 3, figsize=(16, 4 * num_rows))
axes = axes.flatten()

# 3. Loop untuk menggambar boxplot
for i, col in enumerate(numerical_cols):
    # Karena df[col] sudah dipastikan numerik di atas, Anda bisa langsung memplotnya
    sns.boxplot(x=df[col], ax=axes[i], color="lightgreen")
    axes[i].set_title(f"Boxplot of {col}", fontsize=12, fontweight="bold")
    axes[i].set_xlabel("")

# 4. Hapus sisa subplot yang kosong di bagian akhir grid
for j in range(num_features, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle(
    "Deteksi Outlier pada Fitur Numerik Gaya Hidup Mahasiswa",
    fontsize=16,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
plt.show()

Berdasarkan hasil Exploratory Data Analysis (EDA) di notebook saya, diperoleh beberapa insight penting sebagai berikut:

Dataset berisi sekitar 25.500 baris dengan fitur utama berupa kombinasi numerik dan kategorikal: Student_Type, Sleep_Hours, Study_Hours, Social_Media_Hours, Attendance, Exam_Pressure, Family_Support, Month, dan target Stress_Level.
Saya sudah memeriksa missing value dengan df.isnull().sum(), sehingga saya bisa menilai apakah data sudah bersih atau perlu penanganan missing value sebelum preprocessing lebih lanjut.
Distribusi target Stress_Level tidak sepenuhnya seimbang, sehingga model raw cenderung berisiko bias ke kelas mayoritas. Karena itu saya menyiapkan langkah balancing menggunakan teknik resampling.
Visualisasi histogram dan heatmap korelasi membantu mengidentifikasi fitur numerik yang berpotensi penting, terutama Sleep_Hours, Study_Hours, Exam_Pressure, Attendance, dan Family_Support.
Boxplot deteksi outlier menunjukkan adanya nilai ekstrem pada beberapa fitur numerik, yang perlu diperhatikan dan mungkin akan ditangani pada tahap preprocessing berikutnya.
Saya sudah membuat countplot perbandingan sebelum dan sesudah sampling, sehingga bisa memantau secara visual perubahan distribusi kelas yang terjadi setelah penyeimbangan.
Sebagian besar fitur utama sudah berbentuk numerik, sehingga proses scaling dan pelatihan model machine learning akan menjadi lebih mudah setelah preprocessing.
Kesimpulan: EDA saya sudah siap menunjang tahap preprocessing dan modeling berikutnya, terutama dengan fokus pada penyeimbangan kelas dan pemilihan fitur numerik yang relevan.

# **5. Data Preprocessing**

Pada tahap ini, data preprocessing adalah langkah penting untuk memastikan kualitas data sebelum digunakan dalam model machine learning.

Jika Anda menggunakan data teks, data mentah sering kali mengandung nilai kosong, duplikasi, atau rentang nilai yang tidak konsisten, yang dapat memengaruhi kinerja model. Oleh karena itu, proses ini bertujuan untuk membersihkan dan mempersiapkan data agar analisis berjalan optimal.

Berikut adalah tahapan-tahapan yang bisa dilakukan, tetapi **tidak terbatas** pada:
1. Menghapus atau Menangani Data Kosong (Missing Values)
2. Menghapus Data Duplikat
3. Normalisasi atau Standarisasi Fitur
4. Deteksi dan Penanganan Outlier
5. Encoding Data Kategorikal
6. Binning (Pengelompokan Data)

Cukup sesuaikan dengan karakteristik data yang kamu gunakan yah. Khususnya ketika kami menggunakan data tidak terstruktur.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

# 1. Membaca Dataset (Menggunakan relative path mundur 1 folder keluar dari 'prepocessing')
path = '../data_raw/student-lifestyle-and-stress-dataset.csv'
df = pd.read_csv(path)
print(f"Ukuran dataset awal: {df.shape[0]} baris, {df.shape[1]} kolom")

# 2. Pembersihan Target Variable (Stress_Level)
# Baris dengan target 'Stress_Level' yang kosong harus dihapus karena tidak bisa digunakan untuk training
df = df.dropna(subset=['Stress_Level'])
df['Stress_Level'] = df['Stress_Level'].astype(int)

# 3. Menangani Anomali Nilai pada Kolom Attendance
# Mengubah nilai Attendance yang < 0 atau > 100 menjadi NaN agar diisi dengan benar saat tahap imputasi
if 'Attendance' in df.columns:
    df.loc[(df['Attendance'] < 0) | (df['Attendance'] > 100), 'Attendance'] = np.nan

# 4. Imputasi Missing Values (Mengisi Data yang Hilang)
# Memisahkan kolom numerik dan kategorikal
num_cols = ['Sleep_Hours', 'Study_Hours', 'Social_Media_Hours', 'Attendance', 'Exam_Pressure', 'Family_Support', 'Month']
cat_cols = ['Student_Type']

# Imputasi fitur numerik menggunakan MEDIAN agar aman dari pengaruh pencilan (outliers)
imputer_num = SimpleImputer(strategy='median')
df[num_cols] = imputer_num.fit_transform(df[num_cols])

# Imputasi fitur kategorikal menggunakan MODE (nilai yang paling sering muncul)
imputer_cat = SimpleImputer(strategy='most_frequent')
df[cat_cols] = imputer_cat.fit_transform(df[cat_cols])

# 5. Menangani Outliers dengan Metode IQR Capping (Sesuai rencana pada Social_Media_Hours)
def cap_outliers(dataframe, cols):
    dataframe_capped = dataframe.copy()
    for col in cols:
        Q1 = dataframe_capped[col].quantile(0.25)
        Q3 = dataframe_capped[col].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        # Melakukan capping (pembatasan) nilai luar ke batas atas/bawah IQR
        dataframe_capped[col] = np.clip(dataframe_capped[col], lower_bound, upper_bound)
    return dataframe_capped

df = cap_outliers(df, num_cols)

# 6. Encoding Fitur Kategorikal (One-Hot Encoding)
# Mengubah Student_Type menjadi kolom numerik biner (0 atau 1)
df = pd.get_dummies(df, columns=cat_cols, drop_first=True, dtype=int)

# 7. Pemisahan Fitur (X) dan Target (y)
X = df.drop(columns=['Stress_Level'])
y = df['Stress_Level']

# 8. Split Dataset (Train-Test Split)
# Dilakukan SEBELUM scaling untuk mencegah Data Leakage. Menggunakan stratify agar proporsi kelas target seimbang.
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# 9. Feature Scaling (Standardisasi)
# Standardisasi dilakukan hanya pada fitur numerik. Fit dilakukan pada X_train saja.
scaler = StandardScaler()
X_train[num_cols] = scaler.fit_transform(X_train[num_cols])
X_test[num_cols] = scaler.transform(X_test[num_cols])

print("\n--- Preprocessing Selesai ---")
print(f"Ukuran Data Train (X_train): {X_train.shape}")
print(f"Ukuran Data Test (X_test): {X_test.shape}")

output_folder = 'student-lifestyle-preprocessing'
os.makedirs(output_folder, exist_ok=True)

# Menyimpan pecahan data agar siap digunakan langsung oleh folder modelling
X_train.to_csv(f'{output_folder}/X_train.csv', index=False)
X_test.to_csv(f'{output_folder}/X_test.csv', index=False)
y_train.to_csv(f'{output_folder}/y_train.csv', index=False)
y_test.to_csv(f'{output_folder}/y_test.csv', index=False)

print(f"Berhasil mengekspor data ke dalam folder: preprocessing/{output_folder}/")